# 08 v2 · Swiss roll: cierre de la isometría del latente + controles del tutor

Versión 2 del notebook 08. Mantiene **F1–F3** (si adjuntas como *Input* la salida del 08 v1, esas corridas se reutilizan) y añade lo que pidió el tutor:

| | Pregunta | Diseño | Qué se reporta |
|---|---|---|---|
| **G** | Control **sin DMAP**: AE/VAE directo R³ → R² → R³ | `plainae` y `plainvae` (MLP sobre x), sin y con `jac`; 12 réplicas (AE) y 6 (VAE) | las mismas métricas que DMAP + AE/VAE |
| **L** | Error **en el latente** frente a los puntos del oráculo, además del error ambiental | todas las corridas | para cada m: ‖x − x̂‖² (ambiente) **y** ‖r_oráculo − 𝒮(r_latente)‖² (latente), con 𝒮 la mejor transformación de semejanza (E(2) + escala) |
| **C** | Swiss roll sobre una **cuadrícula perfecta** | carta = rejilla regular 136 × 22; β con la misma regla sin etiquetas; DMAP + AE (λ = 0 y 0.1) y control sin DMAP; 6 réplicas (cambian solo las particiones) | figura del latente, anillos y curvas de error |

**Alineación.** r_oráculo es la lectura afín de ψ₁…ψ_m sobre la carta (s, h), en unidades de la carta. El latente de un AE no tiene unidades ni origen, así que la comparación se hace después de la mejor transformación E(2) (traslación, rotación, reflexión) **más una escala isótropa**: Procrustes ortogonal. Sin la escala se compararían números en unidades distintas; con una afín completa se escondería la deformación (estirar un eje). Se reporta también la versión afín como referencia: la diferencia entre las dos mide cuánto del error es anisotropía pura.

**Anillos.** En la figura de anillos, los círculos abiertos son los puntos del oráculo, los puntos llenos son el latente alineado y cada segmento gris une un punto con su pareja. Color = s verdadera.

Duración: ~2 h si adjuntas la salida del 08 v1 y ~5 h sin ella.

## Preparación

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys, zipfile
ROOT = Path('/kaggle/working/dmap_vae_kaggle')
INPUT = Path('/kaggle/input')
CU = 'flash_diffusion.cu'
CODE = ['spectral.py', 'vae.py', 'data.py', 'experiment.py', 'launch.py', 'plots.py',
        'checks.py', 'prepare_alanine.py', 'generate_angles.py', 'SOURCE.json',
        'geometry.py', 'beta_sweep.py', 'protocol.py', 'config_beta_sweep.json',
        'config_beta_sweep_torus.json', 'config_beta_sweep_mnist.json', 'config_beta_sweep_alanine.json', 'latent_figure.py']

# 1) Refresh the code on EVERY run (a partial copy left in /kaggle/working is never trusted).
#    The results in /kaggle/working/results_* are not touched.
sources = sorted({q.parent for q in INPUT.rglob('launch.py') if (q.parent / 'spectral.py').exists()})
archives = sorted(INPUT.rglob('dmap_vae_kaggle*.zip'))
if len(sources) > 1:
    raise FileNotFoundError(f'Hay varias copias del paquete en /kaggle/input: {sources}. Deja solo una.')
ROOT.mkdir(parents=True, exist_ok=True)
if sources:
    shutil.copytree(sources[0], ROOT, dirs_exist_ok=True)
elif len(archives) == 1:
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(ROOT.parent)
else:
    raise FileNotFoundError('Añade el paquete dmap_vae_kaggle como Kaggle Dataset; debe haber una sola copia.')
missing_code = [f for f in CODE if not (ROOT / f).exists()]
assert not missing_code, (f'Faltan archivos del paquete: {missing_code}. Este notebook necesita el '
                          'paquete v5 (dmap_vae_kaggle_v5.zip): sustituye el Dataset anterior.')

# 2) The CUDA source. Kaggle drops or flattens subfolders when a Dataset is built from loose
#    files, so look for it in vendor/, flattened at the root, or inside any attached ZIP.
target = ROOT / 'vendor' / CU
target.parent.mkdir(exist_ok=True)
if not target.exists():
    found = sorted(INPUT.rglob(CU))
    if found:
        shutil.copy2(found[0], target)
    else:
        for arc in sorted(INPUT.rglob('*.zip')):
            with zipfile.ZipFile(arc) as z:
                names = [n for n in z.namelist() if n.endswith('/' + CU) or n == CU]
                if names:
                    target.write_bytes(z.read(names[0]))
                    break
    lic = sorted(INPUT.rglob('LICENSE.FlashDiffusion'))
    if lic and not (target.parent / lic[0].name).exists():
        shutil.copy2(lic[0], target.parent / lic[0].name)
if not target.exists():
    raise FileNotFoundError(
        f'No se encontró {CU} en /kaggle/input. Sube la carpeta vendor/ (o el ZIP completo) '
        'al Kaggle Dataset; al subir archivos sueltos Kaggle omite las subcarpetas.')
expected = json.loads((ROOT / 'SOURCE.json').read_text())['sha256']
actual = hashlib.sha256(target.read_bytes()).hexdigest()
assert actual == expected, f'{CU} no coincide con SOURCE.json ({actual} != {expected}).'
print('Fuente CUDA verificada:', target)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
import importlib.util
missing = [m for m in ['numpy','scipy','pandas','matplotlib','ninja']
           if importlib.util.find_spec(m) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *missing], check=True)
import torch
assert torch.cuda.device_count() >= 2, 'Selecciona GPU T4 x2 en Kaggle.'
print([(i, torch.cuda.get_device_name(i), torch.cuda.get_device_capability(i)) for i in range(2)])
(ROOT / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))


## Controles del paquete (una GPU, ~3 min)

Compila FlashDiffusion y ejecuta `checks.py`: kernel contra matriz densa, convención de signo, métrica kNN, continuación con ceros, paso AE→VAE y los controles nuevos de v5 (regla de degeneración del toro, regla argmin, sonda de etiquetas de MNIST, rotación de trayectorias de alanina). Si falla, no sigas.

In [ ]:
import time
t0 = time.time()
r = subprocess.run([sys.executable, str(ROOT / 'checks.py'), '--backend', 'flash', '--device', 'cuda:0',
                    '--out', str(ROOT / 'checks_08v2.json')],
                   env=dict(os.environ, CUDA_VISIBLE_DEVICES='0'), capture_output=True, text=True)
if r.returncode:
    print(r.stdout[-3000:]); print(r.stderr[-6000:])
    raise RuntimeError('No continuar: falló un control del paquete.')
checks = json.loads((ROOT / 'checks_08v2.json').read_text())
print({k: v for k, v in checks.items() if isinstance(v, str)})
print(f"signo: {checks['sign_overlap_min']:.6f} | kNN suave/plegada: {checks['knn_smooth_ratio']:.2f} / "
      f"{checks['knn_folded_ratio']:.0f} veces el piso | {(time.time()-t0)/60:.1f} min")

## β y resultados previos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
FACTOR = 64
for prev in [Path('/kaggle/working/beta_sweep_swiss/beta_selection.json')] + sorted(Path('/kaggle/input').rglob('beta_sweep_swiss/beta_selection.json')):
    if prev.exists():
        FACTOR = json.loads(prev.read_text())['selected_factor']
        break
print('factor beta =', FACTOR)

def previo(nombre):
    for base in [Path('/kaggle/working')] + [p.parent for p in sorted(Path('/kaggle/input').rglob(nombre))]:
        if (base / nombre / 'metrics.csv').exists():
            return base / nombre
    return None
PREV07 = {0.0: previo('results_swiss_lambda_jac_l0') or previo('results_swiss_latente_R1_sin_reg'),
          0.3: previo('results_swiss_lambda_jac_l0.3')}
print({k: str(v) if v else 'se correrá aquí' for k, v in PREV07.items()})

# ---- beta para la cuadrícula perfecta: misma regla sin etiquetas, propia geometría ----
import protocol as P
cfg_g = json.loads((ROOT / 'config_beta_sweep.json').read_text())
cfg_g.update(grid=True, data_seeds=[7, 1001], modes=[2, 4, 8, 16, 64], out='/kaggle/working/beta_sweep_swiss_grid')
found = sorted(Path('/kaggle/input').rglob('beta_sweep_swiss_grid/beta_selection.json'))
sel_g = json.loads(found[0].read_text()) if found else P.run_sweep(cfg_g, ROOT, 'beta_sweep_swiss_grid')
FACTOR_GRID = sel_g['selected_factor'] or FACTOR
print(f"factor beta cuadrícula = {FACTOR_GRID} (admisibles {sel_g.get('admissible')}); aleatorio = {FACTOR}")

## Plan y estimación de tiempo

El orden es la prioridad: primero F1 (AE, 12 réplicas) y después F3 (VAE). λ = 0 del AE se corre aquí con las 12 réplicas, porque el costo y Fisher necesitan el mismo número de réplicas. Las corridas cuya salida está adjunta como *Input* no se repiten.

In [ ]:
import math
import protocol as P
BASE = json.loads((ROOT / 'config_swiss.json').read_text())
BASE.update(width=60., height=10.)
COMMON = dict(P.COMMON)
LADDER = [2, 4, 8, 16, 64]
M = max(LADDER)
SEEDS_AE = list(range(12))
SEEDS_VAE = list(range(6))
SEEDS_GRID = list(range(6))
comun = dict(beta_factor=FACTOR, modes=LADDER, m_protocols=['scratch', 'warm'], mode_order='spectral',
             oracle_decoder=False, reg_k=10, scratch_first_only=True)
def entrada(tag, models, seeds, lam=0., **extra):
    p = dict(comun, tag=tag, models=models, seeds=seeds, out=f'/kaggle/working/results_swiss_final_{tag}', **extra)
    if lam > 0:
        p.update(reg='jac', reg_weight=lam)
    return p
PLAN = []
# F1-F3 (como en v1; se reutilizan si la salida del 08 v1 está adjunta)
for lam in (0.03, 0.1, 0.0):
    PLAN.append(entrada(f'AE_l{lam:g}', ['ae'], SEEDS_AE, lam))
for lam, kl in ((0.3, 1e-4), (0.3, 1e-3), (0.0, 1e-3)):
    PLAN.append(entrada(f'VAE_l{lam:g}_kl{kl:g}', ['vae'], SEEDS_VAE, lam, kl=kl))
# G: control sin DMAP (MLP sobre x). modes = LADDER solo para tener las filas del oráculo de cada m.
for lam in (0.0, 0.1):
    PLAN.append(entrada(f'PLAIN_AE_l{lam:g}', ['plainae'], SEEDS_AE, lam, m_protocols=['scratch']))
for lam in (0.0, 0.3):
    PLAN.append(entrada(f'PLAIN_VAE_l{lam:g}', ['plainvae'], SEEDS_VAE, lam, m_protocols=['scratch']))
# C: cuadrícula perfecta
for lam in (0.0, 0.1):
    PLAN.append(entrada(f'GRID_AE_l{lam:g}', ['ae'], SEEDS_GRID, lam, grid=True, beta_factor=FACTOR_GRID))
    PLAN.append(entrada(f'GRID_PLAIN_AE_l{lam:g}', ['plainae'], SEEDS_GRID, lam, grid=True, beta_factor=FACTOR_GRID,
                        m_protocols=['scratch']))
# Reutilizar: salida del 08 v1 (o de esta misma versión) adjunta como Input; VAE con kl=1e-3 del 07.
P.reuse_inputs(PLAN)
for p in PLAN:
    if p['tag'] in ('VAE_l0.3_kl0.001', 'VAE_l0_kl0.001') and not (Path(p['out']) / 'metrics.csv').exists():
        alt = PREV07.get(0.3 if 'l0.3' in p['tag'] else 0.0)
        if alt is not None:
            p['out'] = str(alt); print(f"{p['tag']:22s} reutilizado del 07: {alt}")
OUT = {p['tag']: Path(p['out']) for p in PLAN}
AE = {lam: OUT[f'AE_l{lam:g}'] for lam in (0.0, 0.03, 0.1)}
VAE = {(lam, kl): OUT[f'VAE_l{lam:g}_kl{kl:g}'] for lam, kl in ((0.3, 1e-4), (0.3, 1e-3), (0.0, 1e-3))}
SECONDS_PER_MODEL = 80 * 1.4
pendientes = [p for p in PLAN if not (Path(p['out']) / 'metrics.csv').exists()]
P.show_plan(pendientes, 'swiss', SECONDS_PER_MODEL, COMMON) if pendientes else print('todo reutilizado')

## Ejecutar

Se puede re-ejecutar: las configuraciones completas se omiten y las parciales se señalan (bórralas o cambia el `tag`). Para una corrida larga conviene **Save Version → Save & Run All (Commit)**.

In [ ]:
import protocol as P
hechas, problemas = P.run_plan(pendientes, BASE, ROOT, common=COMMON)

## Análisis

Los intervalos del costo son bootstrap (5000 remuestreos de réplicas dentro de cada grupo, razón de medianas). La razón frente al oráculo es **pareada**: misma réplica, mismos datos y misma m.

In [ ]:
from scipy.stats import fisher_exact, mannwhitneyu
import protocol as P
from latent_figure import chart_r2_table
R2S, R2H = .90, .80
rng = np.random.default_rng(0)

def evalua(folder, model):
    d = P.load(folder)
    if d is None:
        return None
    q = P.with_chain_start(d)
    q = q[q.model == model].copy()
    ora = d[d.model == 'oracle'].set_index(['seed', 'm'])
    ref = d[(d.model == 'oracle') & (d.m == M)].set_index('seed').local_distortion
    q = q.merge(chart_r2_table(folder, [model]), on=['seed', 'model', 'm'], how='left')
    q['rect'] = (q.unfolded.astype(bool) & (q.local_distortion <= 2*q.seed.map(ref))
                 & (q.r2_s >= R2S) & (q.r2_h >= R2H))
    q['floor_x'] = [ora.oracle_exact_mse_x.get((s, m), np.nan) for s, m in zip(q.seed, q.m)]
    return q

def boot_ratio(a, b, n=5000):
    a, b = np.asarray(a), np.asarray(b)
    r = [np.median(rng.choice(a, len(a))) / np.median(rng.choice(b, len(b))) for _ in range(n)]
    return np.median(a)/np.median(b), *np.quantile(r, [.025, .975])

def boot_median(x, n=5000):
    x = np.asarray(x)
    return np.median(x), *np.quantile([np.median(rng.choice(x, len(x))) for _ in range(n)], [.025, .975])

def resumen(Q, base, etiqueta):
    filas = []
    for key, q in Q.items():
        if q is None:
            continue
        for m, g in q.groupby('m'):
            b = base[base.m == m].mse_x if base is not None else None
            k, n = int(g.rect.sum()), len(g)
            lo, hi = P.clopper_pearson(k, n)
            c, clo, chi = boot_ratio(g.mse_x, b) if b is not None and len(b) else (np.nan,)*3
            f, flo, fhi = boot_median(g.mse_x / g.floor_x)
            filas.append(dict(grupo=etiqueta, config=key, m=int(m), rectangulo=f'{k}/{n}', tasa=k/n, ic95_inf=lo, ic95_sup=hi,
                              r2_h=g.r2_h.median(), distorsion=g.local_distortion.median(), mse_x=g.mse_x.median(),
                              costo=c, costo_ic=f'{clo:.2f}–{chi:.2f}', vs_oraculo=f, vs_oraculo_ic=f'{flo:.2f}–{fhi:.2f}'))
    return pd.DataFrame(filas)

fmt = {'tasa': '{:.0%}', 'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'r2_h': '{:.3f}', 'distorsion': '{:.3g}',
       'mse_x': '{:.4g}', 'costo': '×{:.2f}', 'vs_oraculo': '×{:.2f}'}

# ---- F1 / F2: AE ----
QA = {lam: evalua(f, 'ae_warm') for lam, f in AE.items()}
TA = resumen(QA, QA.get(0.0), 'AE')
display(TA[TA.m == M].style.format(fmt).set_caption(f'F1/F2 · AE con continuación, m = {M}'))
for lam in (0.03, 0.1):
    a, b = QA.get(lam), QA.get(0.0)
    if a is None or b is None:
        continue
    a, b = a[a.m == M], b[b.m == M]
    ka, kb = int(a.rect.sum()), int(b.rect.sum())
    p = fisher_exact([[ka, len(a)-ka], [kb, len(b)-kb]], alternative='greater')[1]
    c, clo, chi = boot_ratio(a.mse_x, b.mse_x)
    ok = p < .05 and c <= 2
    print(f"F1 λ={lam:g}: rectángulo {ka}/{len(a)} vs {kb}/{len(b)}, Fisher p = {p:.3g}; costo ×{c:.2f} "
          f"(IC {clo:.2f}–{chi:.2f}) -> {'CUMPLE' if ok else 'no cumple'}")
    f, flo, fhi = boot_median(a.mse_x / a.floor_x)
    print(f"F2 λ={lam:g}: MSE_x / MSE_x(oráculo -> roll exacto) = ×{f:.2f} (IC {flo:.2f}–{fhi:.2f}); "
          f"sin regularizar: ×{np.median(b.mse_x/b.floor_x):.2f}")

# ---- F3: VAE, conflicto KL / isometría ----
QV = {k: evalua(f, 'vae_warm') for k, f in VAE.items()}
TV = resumen({f'λ={l:g}, kl={k:g}': q for (l, k), q in QV.items()}, QV.get((0.0, 1e-3)), 'VAE')
display(TV[TV.m == M].style.format(fmt).set_caption(f'F3 · VAE con continuación, m = {M}'))
a, b = QV.get((0.3, 1e-4)), QV.get((0.3, 1e-3))
if a is not None and b is not None:
    a, b = a[a.m == M], b[b.m == M]
    p = mannwhitneyu(a.mse_x, b.mse_x, alternative='less').pvalue
    print(f"F3: λ=0.3 kl=1e-4 MSE_x {a.mse_x.median():.4g} vs kl=1e-3 {b.mse_x.median():.4g}, Mann-Whitney p = {p:.3g}; "
          f"rectángulo {int(a.rect.sum())}/{len(a)} vs {int(b.rect.sum())}/{len(b)} -> "
          f"{'CUMPLE' if p < .05 and a.rect.mean() >= b.rect.mean() else 'no cumple'}")
pd.concat([TA, TV]).to_csv('/kaggle/working/final_tablas.csv', index=False)

In [ ]:
# ---- Figuras para el artículo ----
from latent_figure import latent_grid
rows = [dict(label='DMAP (lectura afín)', folder=AE[0.0], model='oracle'),
        dict(label='AE continuación', folder=AE[0.0], model='ae_warm'),
        dict(label='AE continuación\n+ jac λ=0.03', folder=AE[0.03], model='ae_warm'),
        dict(label='AE continuación\n+ jac λ=0.1', folder=AE[0.1], model='ae_warm')]
latent_grid(rows, LADDER, '/kaggle/working/final_latente',
            'Swiss roll 60×10: latente 2-D alineado afínmente a la carta (test, réplica mediana)')
plt.show()

fig, (a, b) = plt.subplots(1, 2, figsize=(12.5, 4.4))
estilo = {0.0: ('C7', 'o', 'AE continuación'), 0.03: ('C2', 's', 'AE + jac λ=0.03'), 0.1: ('C4', 'D', 'AE + jac λ=0.1')}
for i, (lam, (c, mk, lab)) in enumerate(estilo.items()):
    q = QA.get(lam)
    if q is None:
        continue
    s = q.groupby('m').mse_x
    a.plot(s.median().index, s.median().values, color=c, marker=mk, label=lab)
    a.fill_between(s.median().index, s.quantile(.25).values, s.quantile(.75).values, color=c, alpha=.15, lw=0)
    t = TA[TA.config == lam].sort_values('m')
    b.errorbar(t.m*1.04**(i-1), t.tasa, yerr=[t.tasa-t.ic95_inf, t.ic95_sup-t.tasa], color=c, marker=mk, capsize=3, label=lab)
fl = QA[0.0].groupby('m').floor_x.median()
a.plot(fl.index, fl.values, 'k--', lw=1, label='oráculo afín → roll exacto (piso de la clase)')
a.set(xscale='log', yscale='log', xlabel='m', ylabel='test MSE$_x$ (mediana, IQR)', title='Reconstrucción y piso de la clase afín')
b.set(xscale='log', ylim=(-.05, 1.05), xlabel='m', ylabel='fracción «rectángulo»', title='Rectángulo v2 (IC 95 %, 12 réplicas)')
for ax in (a, b):
    ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off(); ax.legend(fontsize=7)
fig.tight_layout()
fig.savefig('/kaggle/working/final_frontera.png', dpi=200); fig.savefig('/kaggle/working/final_frontera.pdf')
plt.show()

## G y L · Control sin DMAP y error en el espacio latente

Para cada corrida y cada m:
- **ambiente**: ‖x − x̂‖² por coordenada (test);
- **latente vs oráculo**: ‖r_oráculo − 𝒮(r_latente)‖² por coordenada, con 𝒮 de semejanza (E(2) + escala) ajustada sobre los mismos puntos de test (4 parámetros para ~600 puntos);
- **latente vs oráculo (afín)**: lo mismo con una afín completa (6 parámetros). Es más indulgente con la anisotropía;
- **latente vs carta verdadera** y **oráculo vs carta verdadera**, como referencias.

Los modelos sin DMAP no dependen de m: se comparan con el oráculo de cada m (líneas horizontales en la curva ambiental).

In [ ]:
from latent_figure import latent_vs_oracle_table, rings_figure, latent_grid
ETIQ = {'AE_l0': ('DMAP + AE continuación', 'ae_warm'), 'AE_l0.03': ('DMAP + AE cont. + jac 0.03', 'ae_warm'),
        'AE_l0.1': ('DMAP + AE cont. + jac 0.1', 'ae_warm'), 'VAE_l0_kl0.001': ('DMAP + VAE continuación', 'vae_warm'),
        'VAE_l0.3_kl0.001': ('DMAP + VAE cont. + jac 0.3', 'vae_warm'),
        'PLAIN_AE_l0': ('AE sin DMAP', 'plainae'), 'PLAIN_AE_l0.1': ('AE sin DMAP + jac 0.1', 'plainae'),
        'PLAIN_VAE_l0': ('VAE sin DMAP', 'plainvae'), 'PLAIN_VAE_l0.3': ('VAE sin DMAP + jac 0.3', 'plainvae'),
        'GRID_AE_l0': ('cuadrícula · DMAP + AE cont.', 'ae_warm'), 'GRID_AE_l0.1': ('cuadrícula · DMAP + AE cont. + jac 0.1', 'ae_warm'),
        'GRID_PLAIN_AE_l0': ('cuadrícula · AE sin DMAP', 'plainae'), 'GRID_PLAIN_AE_l0.1': ('cuadrícula · AE sin DMAP + jac 0.1', 'plainae')}
LT, AMB = [], []
for tag, (lab, model) in ETIQ.items():
    f = OUT.get(tag)
    d = P.load(f) if f is not None else None
    if d is None:
        continue
    t = latent_vs_oracle_table(f, [model])
    if len(t):
        LT.append(t.assign(corrida=lab, tag=tag))
    q = P.with_chain_start(d)
    q = q[q.model == model]
    AMB.append(q[['seed', 'm', 'mse_x', 'unfolded', 'knn_chart_ratio']].assign(corrida=lab, tag=tag))
LT, AMB = pd.concat(LT), pd.concat(AMB)
ORA = pd.concat([P.load(OUT[t])[lambda d: d.model == 'oracle'][['seed', 'm', 'chart_mse_z', 'oracle_exact_mse_x']].assign(tag=t)
                 for t in ('AE_l0', 'GRID_AE_l0') if P.load(OUT[t]) is not None])
tabla = (LT.groupby(['corrida', 'm'])[['latent_vs_oracle', 'latent_vs_oracle_affine', 'latent_vs_truth', 'oracle_vs_truth']]
           .median().join(AMB.groupby(['corrida', 'm']).mse_x.median()))
orden = [v[0] for v in ETIQ.values()]
tabla = tabla.reindex(sorted(tabla.index, key=lambda k: (orden.index(k[0]), k[1])))
display(tabla.style.format('{:.4g}').set_caption('Medianas por corrida y m: latente (semejanza y afín) y ambiente'))
tabla.to_csv('/kaggle/working/final_v2_latente_vs_oraculo.csv')

In [ ]:
# ---- mse frente al número de modos: ambiente y latente ----
grupos = [('aleatorio', [t for t in ETIQ if not t.startswith('GRID')], 'AE_l0'),
          ('cuadrícula', [t for t in ETIQ if t.startswith('GRID')], 'GRID_AE_l0')]
estilo = {'AE_l0': ('C7', 'o', '-'), 'AE_l0.03': ('C2', 's', '-'), 'AE_l0.1': ('C4', 'D', '-'),
          'VAE_l0_kl0.001': ('C1', 'o', ':'), 'VAE_l0.3_kl0.001': ('C3', 'D', ':'),
          'PLAIN_AE_l0': ('C7', None, '--'), 'PLAIN_AE_l0.1': ('C4', None, '--'),
          'PLAIN_VAE_l0': ('C1', None, '-.'), 'PLAIN_VAE_l0.3': ('C3', None, '-.'),
          'GRID_AE_l0': ('C7', 'o', '-'), 'GRID_AE_l0.1': ('C4', 'D', '-'),
          'GRID_PLAIN_AE_l0': ('C7', None, '--'), 'GRID_PLAIN_AE_l0.1': ('C4', None, '--')}
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for row, (nombre, tags, ref) in enumerate(grupos):
    a, b = axes[row]
    for tag in tags:
        if tag not in set(AMB.tag):
            continue
        c, mk, ls = estilo[tag]
        lab = ETIQ[tag][0]
        g = AMB[AMB.tag == tag].groupby('m').mse_x.median()
        if ETIQ[tag][1] in ('plainae', 'plainvae'):
            a.axhline(g.iloc[0], color=c, ls=ls, lw=1.2, label=lab)
        else:
            a.plot(g.index, g.values, color=c, marker=mk, ls=ls, label=lab)
        l = LT[LT.tag == tag].groupby('m').latent_vs_oracle
        b.plot(l.median().index, l.median().values, color=c, marker=mk or '.', ls=ls, label=lab)
        b.fill_between(l.median().index, l.quantile(.25).values, l.quantile(.75).values, color=c, alpha=.08, lw=0)
    o = ORA[ORA.tag == ref].groupby('m')
    a.plot(o.oracle_exact_mse_x.median().index, o.oracle_exact_mse_x.median().values, 'k--', lw=1,
           label='oráculo afín → roll exacto')
    b.plot(o.chart_mse_z.median().index, o.chart_mse_z.median().values, 'k--', lw=1, label='oráculo vs carta verdadera')
    a.set(xscale='log', yscale='log', xlabel='m (modos DMAP)', ylabel='‖x − x̂‖² (test, mediana)',
          title=f'{nombre}: error ambiental')
    b.set(xscale='log', yscale='log', xlabel='m (modos DMAP)', ylabel='‖r_oráculo − $\mathcal{S}$(r_latente)‖² (mediana, IQR)',
          title=f'{nombre}: error en el latente (alineación E(2) + escala)')
    for ax in (a, b):
        ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off(); ax.legend(fontsize=6)
fig.tight_layout()
fig.savefig('/kaggle/working/final_v2_mse_vs_modos.png', dpi=200); fig.savefig('/kaggle/working/final_v2_mse_vs_modos.pdf')
plt.show()

# ---- anillos: puntos del oráculo (círculos) y latente alineado (puntos) ----
for nombre, tags in (('aleatorio', ['AE_l0', 'AE_l0.1', 'PLAIN_AE_l0', 'PLAIN_AE_l0.1']),
                     ('cuadricula', ['GRID_AE_l0', 'GRID_AE_l0.1', 'GRID_PLAIN_AE_l0', 'GRID_PLAIN_AE_l0.1'])):
    rows = [dict(label=ETIQ[t][0], folder=OUT[t], model=ETIQ[t][1]) for t in tags if P.load(OUT[t]) is not None]
    if rows:
        rings_figure(rows, [4, 8, 16, 64], f'/kaggle/working/final_v2_anillos_{nombre}',
                     f'Swiss roll ({nombre}): puntos del oráculo (círculos) y latente alineado por semejanza (puntos); '
                     'ejes no isótropos')
        plt.show()

# ---- figura del latente para la cuadrícula perfecta ----
rows = [dict(label='DMAP (lectura afín)', folder=OUT['GRID_AE_l0'], model='oracle')] + \
       [dict(label=ETIQ[t][0].replace('cuadrícula · ', ''), folder=OUT[t], model=ETIQ[t][1])
        for t in ('GRID_AE_l0', 'GRID_AE_l0.1', 'GRID_PLAIN_AE_l0', 'GRID_PLAIN_AE_l0.1') if P.load(OUT[t]) is not None]
latent_grid(rows, LADDER, '/kaggle/working/final_v2_latente_cuadricula',
            'Swiss roll sobre cuadrícula perfecta: latente 2-D alineado afínmente a la carta (test, réplica mediana)')
plt.show()
rows = [dict(label='DMAP (lectura afín)', folder=OUT['AE_l0'], model='oracle')] + \
       [dict(label=ETIQ[t][0], folder=OUT[t], model=ETIQ[t][1]) for t in ('PLAIN_AE_l0', 'PLAIN_AE_l0.1', 'PLAIN_VAE_l0', 'PLAIN_VAE_l0.3')
        if P.load(OUT[t]) is not None]
latent_grid(rows, LADDER, '/kaggle/working/final_v2_latente_sin_dmap',
            'Control sin DMAP (MLP sobre x): el latente no depende de m; se repite en cada columna')
plt.show()

# ---- tasas de rectángulo v2 del control sin DMAP y de la cuadrícula ----
from latent_figure import chart_r2_table
filas = []
for tag in [t for t in ETIQ if t.startswith(('PLAIN', 'GRID'))]:
    if P.load(OUT[tag]) is None:
        continue
    model = ETIQ[tag][1]
    q = evalua(OUT[tag], model)
    q = q[q.m == (q.m.min() if model in ('plainae', 'plainvae') else M)]
    k, n = int(q.rect.sum()), len(q)
    lo, hi = P.clopper_pearson(k, n)
    filas.append(dict(corrida=ETIQ[tag][0], rectangulo=f'{k}/{n}', ic95_inf=lo, ic95_sup=hi,
                      desplegadas=f"{int(q.unfolded.astype(bool).sum())}/{n}", r2_h=q.r2_h.median(),
                      distorsion=q.local_distortion.median(), mse_x=q.mse_x.median()))
display(pd.DataFrame(filas).style.format({'ic95_inf': '{:.2f}', 'ic95_sup': '{:.2f}', 'r2_h': '{:.3f}',
                                          'distorsion': '{:.3g}', 'mse_x': '{:.4g}'})
        .set_caption('Rectángulo v2: control sin DMAP y cuadrícula (DMAP en m = M)'))

## Tabla normalizada con intervalos y figuras para el artículo (sin reentrenar)

- **Normalización por eje**: error / Var(eje) de la carta verdadera (test). 0 = perfecto, 1 = lo mismo que predecir la media (sin información de ese eje); > 1 = peor que la media, lo que ocurre cuando, con una sola escala común, el latente invade el eje h con la deformación del eje s (el arco). Así un colapso de la altura (Var h ≈ Var s / 36) no queda escondido por el eje s.
- **Intervalos**: bootstrap al 95 % de la mediana sobre réplicas (5000 remuestreos).
- **Columna afín**: no informa para m ≤ 2 (latente y oráculo son ambos imágenes afines de ψ₁, ψ₂), así que ahí queda vacía.
- **Alineación**: semejanza (E(2) + escala) ajustada sobre los mismos puntos de test (4 parámetros para ~600 puntos).

In [ ]:
rng_b = np.random.default_rng(0)
def mediana_ic(x, n=5000):
    x = np.asarray(x, float); x = x[np.isfinite(x)]
    if not len(x):
        return np.nan, np.nan, np.nan
    b = np.median(rng_b.choice(x, (n, len(x))), axis=1)
    return np.median(x), *np.quantile(b, [.025, .975])

COLS = [('latent_vs_oracle_s_norm', 'lat↔oráculo s'), ('latent_vs_oracle_h_norm', 'lat↔oráculo h'),
        ('latent_vs_truth_s_norm', 'lat↔verdad s'), ('latent_vs_truth_h_norm', 'lat↔verdad h'),
        ('oracle_vs_truth_s_norm', 'oráculo↔verdad s'), ('oracle_vs_truth_h_norm', 'oráculo↔verdad h'),
        ('latent_vs_oracle', 'lat↔oráculo (unid. carta)'), ('latent_vs_oracle_affine', 'lat↔oráculo afín')]
filas = []
for (lab, m), g in LT.groupby(['corrida', 'm']):
    fila = dict(corrida=lab, m=int(m), n=g.seed.nunique())
    for c, nombre in COLS:
        med, lo, hi = mediana_ic(g[c])
        fila[nombre] = med; fila[nombre + ' IC'] = f'{lo:.3g}–{hi:.3g}' if np.isfinite(lo) else ''
    a = AMB[(AMB.corrida == lab) & (AMB.m == (AMB[AMB.corrida == lab].m.min() if lab.find('sin DMAP') >= 0 else m))]
    med, lo, hi = mediana_ic(a.mse_x)
    fila['‖x−x̂‖²'] = med; fila['‖x−x̂‖² IC'] = f'{lo:.3g}–{hi:.3g}'
    filas.append(fila)
TN = pd.DataFrame(filas)
TN = TN.iloc[sorted(range(len(TN)), key=lambda i: (orden.index(TN.corrida.iloc[i]), TN.m.iloc[i]))].reset_index(drop=True)
TN.to_csv('/kaggle/working/final_v2_tabla_normalizada.csv', index=False)
num = [n for _, n in COLS] + ['‖x−x̂‖²']
display(TN[TN.m == M][['corrida', 'n'] + [c for n in num for c in (n, n + ' IC')]]
        .style.format({n: '{:.3g}' for n in num}, na_rep='—')
        .set_caption(f'm = {M}: medianas e IC 95 % (bootstrap sobre réplicas). Ejes s y h normalizados por su varianza.'))

In [ ]:
# ---- Figura 1 del artículo: anillos (3 modelos x 3 presupuestos) ----
rows = [dict(label='DMAP + AE\ncontinuación', folder=OUT['AE_l0'], model='ae_warm'),
        dict(label='DMAP + AE cont.\n+ jac λ=0.1', folder=OUT['AE_l0.1'], model='ae_warm'),
        dict(label='AE sin DMAP\n(R³→R²→R³)', folder=OUT['PLAIN_AE_l0'], model='plainae', oracle_folder=OUT['PLAIN_AE_l0'])]
rings_figure(rows, [8, 16, 64], '/kaggle/working/final_articulo_anillos',
             'Puntos del oráculo DMAP (círculos) y latente 2-D alineado por semejanza E(2)+escala (puntos); color = s')
plt.show()

# ---- Figura 2 del artículo: error ambiental y error en el latente por eje, frente a m ----
sel = [('AE_l0', 'DMAP + AE continuación', 'C7', 'o', '-'), ('AE_l0.03', 'DMAP + AE cont. + jac 0.03', 'C2', 's', '-'),
       ('AE_l0.1', 'DMAP + AE cont. + jac 0.1', 'C4', 'D', '-'), ('VAE_l0.3_kl0.001', 'DMAP + VAE cont. + jac 0.3', 'C3', 'v', ':'),
       ('PLAIN_AE_l0', 'AE sin DMAP', 'C7', None, '--'), ('PLAIN_VAE_l0', 'VAE sin DMAP', 'C1', None, '--')]
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for tag, lab, c, mk, ls in sel:
    if tag not in set(LT.tag):
        continue
    plain = ETIQ[tag][1] in ('plainae', 'plainvae')
    a = AMB[AMB.tag == tag]
    if plain:
        axes[0].axhline(a.mse_x.median(), color=c, ls=ls, lw=1.2, label=lab)
    else:
        s = a.groupby('m').mse_x
        axes[0].plot(s.median().index, s.median().values, color=c, marker=mk, ls=ls, label=lab)
        axes[0].fill_between(s.median().index, s.quantile(.25).values, s.quantile(.75).values, color=c, alpha=.12, lw=0)
    for ax, col in ((axes[1], 'latent_vs_oracle_s_norm'), (axes[2], 'latent_vs_oracle_h_norm')):
        g = LT[LT.tag == tag].groupby('m')[col]
        ax.plot(g.median().index, g.median().values, color=c, marker=mk or '.', ls=ls, label=lab)
        ax.fill_between(g.median().index, g.quantile(.25).values, g.quantile(.75).values, color=c, alpha=.12, lw=0)
o = ORA[ORA.tag == 'AE_l0'].groupby('m').oracle_exact_mse_x.median()
axes[0].plot(o.index, o.values, 'k--', lw=1, label='oráculo afín → roll exacto')
ref = LT[LT.tag == 'AE_l0'].groupby('m')
axes[1].plot(ref.oracle_vs_truth_s_norm.median().index, ref.oracle_vs_truth_s_norm.median().values, 'k--', lw=1, label='oráculo vs carta verdadera')
axes[2].plot(ref.oracle_vs_truth_h_norm.median().index, ref.oracle_vs_truth_h_norm.median().values, 'k--', lw=1, label='oráculo vs carta verdadera')
axes[0].set(ylabel='‖x − x̂‖² por coordenada (test)', title='Error ambiental')
axes[1].set(ylabel='error / Var(s)', title='Latente vs puntos del oráculo: eje s')
axes[2].set(ylabel='error / Var(h)', title='Latente vs puntos del oráculo: eje h')
for ax in axes:
    ax.set(xscale='log', yscale='log', xlabel='m (modos DMAP)')
    ax.set_xticks(LADDER); ax.set_xticklabels([str(m) for m in LADDER]); ax.minorticks_off()
for ax in axes[1:]:
    ax.axhline(1, color='0.6', lw=.8, ls=':')
    ax.text(LADDER[0], 1.08, '= predecir la media', fontsize=7, color='0.4')
axes[0].legend(fontsize=7)
fig.suptitle('Mediana e IQR sobre réplicas. Latente alineado por semejanza E(2)+escala; ejes normalizados por su varianza.', fontsize=9)
fig.tight_layout()
fig.savefig('/kaggle/working/final_articulo_errores.png', dpi=200); fig.savefig('/kaggle/working/final_articulo_errores.pdf')
plt.show()

## Qué descargar

`/kaggle/working/results_swiss_final_*`, `beta_sweep_swiss_grid/`, `final_latente.*`, `final_frontera.*`, `final_v2_mse_vs_modos.*`, `final_v2_anillos_*.*`, `final_v2_latente_cuadricula.*`, `final_v2_latente_sin_dmap.*`, `final_articulo_anillos.*`, `final_articulo_errores.*`, `final_tablas.csv`, `final_v2_latente_vs_oraculo.csv` y `final_v2_tabla_normalizada.csv`.

**Para regenerar solo las tablas y figuras sin reentrenar:** adjunta como *Input* la salida de esta corrida y ejecuta todo; las corridas se reutilizan.